In [ ]:
# =========================================================
# FINAL DATASET PIPELINE (MERGE → FILTER → SPLIT → BALANCE)
# =========================================================

import random
import shutil
import uuid
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.model_selection import train_test_split

# =========================================================
# PATHS (CHANGE IF NEEDED)
# =========================================================
SRC_ROOT = Path("/kaggle/input/datasets/kelkalot/the-hyper-kvasir-dataset/labeled-images")
BASE_DIR = Path("/kaggle/working/final_clean_dataset")

if BASE_DIR.exists():
    shutil.rmtree(BASE_DIR)

TRAIN_DIR = BASE_DIR / "train"
TEST_DIR  = BASE_DIR / "test"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# SETTINGS
# =========================================================
TARGET_PER_CLASS = 868
REMOVE_THRESHOLD = 100
TEST_SIZE = 0.25
RANDOM_STATE = 42

random.seed(RANDOM_STATE)

# =========================================================
# AUGMENTATION (SAFE)
# =========================================================
def augment_pil(img):
    img = img.convert("RGB")

    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)

    img = img.rotate(random.uniform(-25, 25))

    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.15))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.85, 1.15))
    img = ImageEnhance.Color(img).enhance(random.uniform(0.9, 1.1))

    if random.random() < 0.3:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.2)))

    return img

# =========================================================
# STEP 1: SCAN + MERGE
# =========================================================
print("\n📊 STEP 1: SCANNING + MERGING\n")

valid_classes = {}

for level1 in SRC_ROOT.iterdir():
    if not level1.is_dir():
        continue

    for level2 in level1.iterdir():
        if not level2.is_dir():
            continue

        for cls in level2.iterdir():
            if not cls.is_dir():
                continue

            imgs = [p for p in cls.glob("*") if p.suffix.lower() in [".jpg", ".png", ".jpeg"]]
            n = len(imgs)

            cls_lower = cls.name.lower()

            # =========================================================
            # MERGE RULES
            # =========================================================
            if "ulcerative-colitis-grade" in cls_lower:
                final_cls = "ulcerative-colitis"

            elif "esophagitis" in cls_lower:
                final_cls = "esophagitis"

            else:
                final_cls = cls.name

            class_name = final_cls

            if class_name not in valid_classes:
                valid_classes[class_name] = []

            valid_classes[class_name].extend(imgs)

            print(f"📂 {cls.name} → {class_name} ({n})")

# =========================================================
# STEP 2: REMOVE SMALL CLASSES (AFTER MERGE)
# =========================================================
print("\n🧹 STEP 2: REMOVING SMALL CLASSES\n")

filtered_classes = {}

for cls_name, imgs in valid_classes.items():
    n = len(imgs)

    if n < REMOVE_THRESHOLD:
        print(f"❌ Removed: {cls_name} ({n})")
    else:
        filtered_classes[cls_name] = imgs
        print(f"✅ Kept: {cls_name} ({n})")

valid_classes = filtered_classes

# =========================================================
# STEP 3: TRAIN / TEST SPLIT
# =========================================================
print("\n📊 STEP 3: SPLITTING\n")

for cls_name, imgs in valid_classes.items():

    train, test = train_test_split(
        imgs, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    train_dir = TRAIN_DIR / cls_name
    test_dir  = TEST_DIR / cls_name

    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    for p in train:
        shutil.copy(p, train_dir / p.name)

    for p in test:
        shutil.copy(p, test_dir / p.name)

# =========================================================
# STEP 4: BALANCE TRAIN (AUGMENTATION)
# =========================================================
print("\n🔄 STEP 4: BALANCING TRAIN DATA\n")

# =========================================================
# STEP 4: AUGMENT ONLY MINORITY CLASSES
# =========================================================
print("\n🔄 STEP 4: AUGMENTING ONLY MINORITY CLASSES\n")

MINORITY_THRESHOLD = 900   # you can tune this
TARGET_PER_CLASS   = 900   # only for minority classes

for cls_dir in TRAIN_DIR.iterdir():
    imgs = list(cls_dir.glob("*"))
    n = len(imgs)

    print(f"Class: {cls_dir.name} | Original: {n}")

    # ✅ ONLY AUGMENT if minority
    if n < MINORITY_THRESHOLD:
        needed = TARGET_PER_CLASS - n

        print(f"   ➕ Augmenting {needed} images")

        for _ in range(needed):
            src = random.choice(imgs)

            try:
                img = Image.open(src)
                aug = augment_pil(img)

                name = f"{src.stem}_aug_{uuid.uuid4().hex[:8]}.jpg"
                aug.save(cls_dir / name, quality=95)

            except:
                continue

    else:
        print("   ✅ Majority class → No augmentation (kept as-is)")

# =========================================================
# STEP 5: FINAL PRINT
# =========================================================
print("\n📊 FINAL TRAIN DATASET (AFTER AUGMENTATION)\n")

for cls_dir in TRAIN_DIR.iterdir():
    print(f"{cls_dir.name}: {len(list(cls_dir.glob('*')))}")

print("\n📊 FINAL TEST DATASET\n")

for cls_dir in TEST_DIR.iterdir():
    print(f"{cls_dir.name}: {len(list(cls_dir.glob('*')))}")

print("\n✅ DATASET READY →", BASE_DIR)

In [ ]:
import shutil

shutil.make_archive(
    '/kaggle/working/final_clean_dataset',
    'zip',
    '/kaggle/working/final_clean_dataset'
)

In [ ]:
# =========================================================
# HEASNet v5 — CORRECTED VERSION (ALL BUGS FIXED)
# =========================================================
# FIXES APPLIED:
#   [1] FocalLoss is no longer overwritten — used as the single criterion
#   [2] GradCAM target layer corrected to model.win.conv (exists in HEASNet)
#   [3] Validation split now uses a separate ImageFolder with test_tfm
#   [4] TTA operates on raw (un-normalized) PIL images, not normalized tensors
#   [5] Grad-CAM visualization de-normalizes image before display
#   [6] CosineAnnealingLR scheduler added

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import cv2

from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_curve, f1_score
)
from sklearn.preprocessing import label_binarize
from thop import profile

# ================= CONFIG =================
IMG_SIZE    = 224
BATCH_SIZE  = 16
EPOCHS      = 40
LR          = 1e-4
PATIENCE    = 40
TTA_STEPS   = 5

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BASE_DIR = Path("/kaggle/working/final_clean_dataset")

# ImageNet normalization constants
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ================= TRANSFORMS =================
train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandAugment(2, 7),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# FIX [3]: val/test use the same clean transform (no augmentation)
eval_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# FIX [4]: TTA transform applied to raw PIL images BEFORE normalization
#           Normalization is done separately after augmentation
tta_aug_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),          # converts to [0,1] tensor
    transforms.Normalize(MEAN, STD) # normalize after augmentation
])

# ================= DATA =================
# Full training folder with augmentation
full_train = datasets.ImageFolder(BASE_DIR / "train", transform=train_tfm)

NUM_CLASSES = len(full_train.classes)
CLASS_NAMES = full_train.classes

# FIX [3]: Create a CLEAN copy of the training folder for the val split
#          so val images are not augmented
full_train_clean = datasets.ImageFolder(BASE_DIR / "train", transform=eval_tfm)

# Use the same random indices for both views
val_ratio  = 0.2
val_size   = int(val_ratio * len(full_train))
train_size = len(full_train) - val_size

indices        = torch.randperm(len(full_train)).tolist()
train_indices  = indices[:train_size]
val_indices    = indices[train_size:]

train_ds = Subset(full_train,       train_indices)   # augmented
val_ds   = Subset(full_train_clean, val_indices)     # clean — FIX [3]
test_ds  = datasets.ImageFolder(BASE_DIR / "test", transform=eval_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# ================= LOSS =================
# FIX [1]: FocalLoss defined ONCE and used as the single criterion
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none',
                             label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

criterion = FocalLoss(gamma=2.0, label_smoothing=0.1)   # only definition — never overwritten

# ================= MODEL =================
class SEBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, max(c // 16, 4), 1),
            nn.ReLU(),
            nn.Conv2d(max(c // 16, 4), c, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x)


class WindowAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)  # FIX [2]: referenced as model.win.conv in GradCAM
        self.bn   = nn.BatchNorm2d(dim)
        self.act  = nn.GELU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class AxialAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.qkv  = nn.Conv2d(dim, dim * 3, 1)
        self.proj = nn.Conv2d(dim, dim, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        qkv     = self.qkv(x).reshape(B, 3, C, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        q, k, v = q.permute(0, 2, 1), k.permute(0, 2, 1), v.permute(0, 2, 1)
        attn    = torch.softmax(q @ k.transpose(-1, -2) / (C ** 0.5), dim=-1)
        out     = (attn @ v).permute(0, 2, 1).reshape(B, C, H, W)
        return self.proj(out)


class HEASNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_m", pretrained=True,
            features_only=True, out_indices=(4,)
        )
        ch = self.backbone.feature_info.channels()[-1]

        self.proj = nn.Conv2d(ch, 256, 1)
        self.se   = SEBlock(256)
        self.win  = WindowAttention(256)   # model.win.conv is the GradCAM target
        self.ax   = AxialAttention(256)
        self.norm = nn.BatchNorm2d(256)
        self.pool = nn.AdaptiveAvgPool2d(1)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.backbone(x)[-1]
        x = self.proj(x)
        x = x + self.se(x)
        x = x + self.win(x)
        x = x + self.ax(x)
        x = self.norm(x)
        x = self.pool(x).flatten(1)
        return self.head(x)


model     = HEASNet(NUM_CLASSES).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# FIX [6]: Cosine annealing LR scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

# ================= TRAIN FUNCTION =================
def run_epoch(loader, train=True):
    model.train() if train else model.eval()

    preds, labels, probs_list = [], [], []
    total_loss, correct, total = 0, 0, 0

    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            if train:
                optimizer.zero_grad()

            out  = model(x)
            loss = criterion(out, y)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            p    = torch.softmax(out, 1)
            pred = out.argmax(1)

            total_loss += loss.item()
            correct    += (pred == y).sum().item()
            total      += y.size(0)

            preds.extend(pred.detach().cpu().numpy())
            labels.extend(y.detach().cpu().numpy())
            probs_list.extend(p.detach().cpu().numpy())

    return (
        correct / total,
        total_loss / len(loader),
        np.array(preds),
        np.array(labels),
        np.array(probs_list)
    )

# ================= TRAIN LOOP =================
best_f1  = 0
patience = 0

# History trackers for plotting
history = {
    "train_loss": [], "val_loss": [],
    "train_acc":  [], "val_acc":  []
}

print(f"\n{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7} | {'Macro-F1':>8} | {'LR':>9}")
print("-" * 75)

for epoch in range(EPOCHS):
    tr_acc,  tr_loss,  _,         _,        _  = run_epoch(train_loader, train=True)
    val_acc, val_loss, val_preds, val_lbls,  _  = run_epoch(val_loader,   train=False)

    f1 = f1_score(val_lbls, val_preds, average='macro')

    # FIX [6]: step the scheduler every epoch
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # Store history
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(val_acc)

    # Clean per-epoch print
    print(f"{epoch+1:>6} | {tr_loss:>10.4f} | {tr_acc:>9.4f} | {val_loss:>8.4f} | {val_acc:>7.4f} | {f1:>8.4f} | {current_lr:>9.2e}")

    if f1 > best_f1:
        best_f1  = f1
        patience = 0
        torch.save(model.state_dict(), "best_model.pth")
        print(f"{'':>6}   ✅ Best model saved (F1 = {best_f1:.4f})")
    else:
        patience += 1
        if patience >= PATIENCE:
            print(f"{'':>6}   ⛔ Early stopping at epoch {epoch+1}")
            break

# ================= TRAINING CURVES PLOT =================
epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Loss plot ---
ax1.plot(epochs_ran, history["train_loss"], label="Train Loss", color="#E74C3C", linewidth=2, marker='o', markersize=3)
ax1.plot(epochs_ran, history["val_loss"],   label="Val Loss",   color="#3498DB", linewidth=2, marker='o', markersize=3)
ax1.set_title("Loss per Epoch",     fontsize=14, fontweight='bold')
ax1.set_xlabel("Epoch",             fontsize=12)
ax1.set_ylabel("Loss",              fontsize=12)
ax1.legend(fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.5)

# --- Accuracy plot ---
ax2.plot(epochs_ran, history["train_acc"], label="Train Acc", color="#E74C3C", linewidth=2, marker='o', markersize=3)
ax2.plot(epochs_ran, history["val_acc"],   label="Val Acc",   color="#3498DB", linewidth=2, marker='o', markersize=3)
ax2.set_title("Accuracy per Epoch", fontsize=14, fontweight='bold')
ax2.set_xlabel("Epoch",             fontsize=12)
ax2.set_ylabel("Accuracy",          fontsize=12)
ax2.set_ylim(0, 1)
ax2.legend(fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.suptitle("HEASNet — Training Curves", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Training curves saved to training_curves.png")

# ================= LOAD BEST =================
model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
print(f"\nLoaded best model (Val Macro-F1 = {best_f1:.4f})")

# ================= TTA INFERENCE =================
# FIX [4]: TTA is applied to the raw PIL image, then normalized inside tta_aug_tfm
#           We use a separate raw dataset (no normalization) for TTA
raw_test_ds = datasets.ImageFolder(
    BASE_DIR / "test",
    transform=transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE))])  # PIL only
)

def tta_predict(raw_dataset):
    """Run TTA on raw PIL images — augmentation + normalization applied per step."""
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for idx in range(len(raw_dataset)):
            pil_img, label = raw_dataset[idx]
            probs_sum = torch.zeros(NUM_CLASSES, device=DEVICE)

            for _ in range(TTA_STEPS):
                tensor = tta_aug_tfm(pil_img).unsqueeze(0).to(DEVICE)  # augment then normalize
                probs_sum += torch.softmax(model(tensor), dim=1).squeeze(0)

            all_probs.append((probs_sum / TTA_STEPS).cpu().numpy())
            all_labels.append(label)

    return np.array(all_probs), np.array(all_labels)

print("\nRunning TTA inference on test set...")
probs, labels = tta_predict(raw_test_ds)
preds = probs.argmax(1)

# ================= METRICS =================
print("\n" + "="*50)
print("FINAL TEST RESULTS")
print("="*50)
print(f"Accuracy : {accuracy_score(labels, preds):.4f}")
print(f"Kappa    : {cohen_kappa_score(labels, preds):.4f}")
print(f"MCC      : {matthews_corrcoef(labels, preds):.4f}")
print()
print(classification_report(labels, preds, target_names=CLASS_NAMES , digits=4))

# ================= ROC CURVE =================
lb = label_binarize(labels, classes=list(range(NUM_CLASSES)))

plt.figure(figsize=(8, 6))
for i in range(NUM_CLASSES):
    fpr, tpr, _ = roc_curve(lb[:, i], probs[:, i])
    plt.plot(fpr, tpr, label=CLASS_NAMES[i])

plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multi-class ROC Curve")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ================= CONFUSION MATRIX =================
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

# ================= FLOPs & PARAMS =================
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
flops, params = profile(model, inputs=(dummy,), verbose=False)
print(f"\nFLOPs  : {flops / 1e9:.2f} GFLOPs")
print(f"Params : {params / 1e6:.2f} M")

# ================= LATENCY =================
model.eval()
# Warm-up
for _ in range(5):
    _ = model(dummy)

start = time.time()
for _ in range(50):
    _ = model(dummy)
end = time.time()
print(f"Avg Inference Time : {(end - start) / 50 * 1000:.2f} ms/image")

# ================= GRAD-CAM =================
class GradCAM:
    def __init__(self, model):
        self.model       = model
        self.activations = None
        self.gradients   = None

        # FIX [2]: correct attribute path — model.win.conv exists in HEASNet
        self.target_layer = model.win.conv

        self.target_layer.register_forward_hook(self._save_activations)
        self.target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, x, class_idx):
        self.model.eval()
        out = self.model(x)

        self.model.zero_grad()
        out[:, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1)
        cam     = torch.relu(cam).squeeze().cpu().numpy()
        cam     = cam / (cam.max() + 1e-8)

        return cam


def denormalize(tensor, mean=MEAN, std=STD):
    """FIX [5]: Reverse ImageNet normalization for display."""
    t = tensor.clone().cpu()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return t.permute(1, 2, 0).numpy().clip(0, 1)


gradcam = GradCAM(model)

# Visualize first test image
img_tensor, label = test_ds[0]
inp = img_tensor.unsqueeze(0).to(DEVICE)
inp.requires_grad_(True)

cam     = gradcam.generate(inp, label)
cam     = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))

img_np  = denormalize(img_tensor)   # FIX [5]: de-normalized for display

plt.figure(figsize=(6, 6))
plt.imshow(img_np)
plt.imshow(cam, cmap='jet', alpha=0.45)
plt.title(f"Grad-CAM — Ground Truth: {CLASS_NAMES[label]}")
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 🔥 FINAL HEASNet + ADVANCED INTERPRETABILITY (ALL-IN-ONE)
# =========================================================

# !pip install timm fvcore reportlab -q

import os, timm, torch, torch.nn as nn, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from scipy.stats import pearsonr, spearmanr
from torchvision import datasets, transforms
from sklearn.metrics import *
from sklearn.preprocessing import label_binarize
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

# reportlab for PDF
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable, PageBreak, Image as RLImage, KeepTogether
)

# FLOPs
try:
    from fvcore.nn import FlopCountAnalysis, parameter_count_table
    FVCORE = True
except ImportError:
    FVCORE = False
    print("⚠️  fvcore not found – FLOPs will be estimated manually")

# ================= SETTINGS =================
MODEL_PATH = "/kaggle/input/datasets/visheshtanwar26/gascode/best_model (2).pth"
TEST_DIR   = "/kaggle/working/final_clean_dataset/test"
SAVE_DIR   = "/kaggle/working/interpretability"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_SIZE  = 224
TTA_ROUNDS  = 5
LATENCY_RUNS = 50   # warm-up + measure

print("Using:", DEVICE)


# =========================================================
# ================= MODEL =================
# =========================================================

class SEBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c, max(c // 16, 4), 1),
            nn.ReLU(),
            nn.Conv2d(max(c // 16, 4), c, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return x * self.se(x)

class WindowAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.bn   = nn.BatchNorm2d(dim)
        self.act  = nn.GELU()
    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class AxialAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.qkv  = nn.Conv2d(dim, dim * 3, 1)
        self.proj = nn.Conv2d(dim, dim, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(x).reshape(B, 3, C, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        q, k, v = q.permute(0, 2, 1), k.permute(0, 2, 1), v.permute(0, 2, 1)
        attn = torch.softmax(q @ k.transpose(-1, -2) / (C ** 0.5), dim=-1)
        out = (attn @ v).permute(0, 2, 1).reshape(B, C, H, W)
        return self.proj(out)

class HEASNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.backbone = timm.create_model(
            "tf_efficientnetv2_m",
            pretrained=False,
            features_only=True,
            out_indices=(4,)
        )
        ch = self.backbone.feature_info.channels()[-1]
        self.proj = nn.Conv2d(ch, 256, 1)
        self.se   = SEBlock(256)
        self.win  = WindowAttention(256)
        self.ax   = AxialAttention(256)
        self.norm = nn.BatchNorm2d(256)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = self.backbone(x)[-1]
        x = self.proj(x)
        x = x + self.se(x)
        x = x + self.win(x)
        x = x + self.ax(x)
        x = self.norm(x)
        x = self.pool(x).flatten(1)
        return self.head(x)


# ================= DATA =================
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

dataset     = datasets.ImageFolder(TEST_DIR, transform=transform)
class_names = dataset.classes
NUM_CLASSES = len(class_names)

# ================= LOAD MODEL =================
model = HEASNet(NUM_CLASSES).to(DEVICE)
ckpt  = torch.load(MODEL_PATH, map_location=DEVICE)
if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    model.load_state_dict(ckpt["model_state_dict"])
else:
    model.load_state_dict(ckpt)
model.eval()
print("✅ Model Loaded")


# =========================================================
# 🔥 GRAD-CAM   (BUG FIX: .detach() before .numpy())
# =========================================================

class GradCAM:
    def __init__(self, mdl):
        self.gradients  = None
        self.activations = None

        def f_hook(m, i, o):
            self.activations = o.detach()          # ← detach activations

        def b_hook(m, gi, go):
            self.gradients = go[0].detach()        # ← detach gradients

        mdl.win.conv.register_forward_hook(f_hook)
        # FIX: use register_full_backward_hook to silence FutureWarning
        mdl.win.conv.register_full_backward_hook(b_hook)

    def generate(self, x, cls):
        model.zero_grad()
        out = model(x)
        out[0, cls].backward()

        g   = self.gradients[0]          # (C, H, W)
        a   = self.activations[0]        # (C, H, W)
        w   = g.mean((1, 2))
        cam = (w[:, None, None] * a).sum(0)
        cam = torch.relu(cam)
        # ✅ BUG FIX: was missing .detach() → RuntimeError fixed
        cam = cam.detach()
        mx  = cam.max()
        if mx > 0:
            cam = cam / mx
        return cam.cpu().numpy()


# =========================================================
# 🔥 ATTENTION HOOKS
# =========================================================

class AttentionCapture:
    def __init__(self, mdl):
        self.maps = {}
        mdl.se.register_forward_hook( lambda m, i, o: self._save("SE",  o))
        mdl.win.register_forward_hook(lambda m, i, o: self._save("WIN", o))
        mdl.ax.register_forward_hook( lambda m, i, o: self._save("AX",  o))

    def _save(self, name, x):
        self.maps[name] = x[0].mean(0).detach().cpu().numpy()

    def get(self):
        return dict(self.maps)


# =========================================================
# 🔥 VISUALIZATION
# =========================================================

def normalize_map(m):
    mn, mx = m.min(), m.max()
    return (m - mn) / (mx - mn + 1e-8)

def resize_map(m, size):
    from PIL import Image as PILImage
    arr = (normalize_map(m) * 255).astype(np.uint8)
    img = PILImage.fromarray(arr).resize(size, PILImage.BILINEAR)
    return np.array(img) / 255.0

def plot_all(img_pil, maps, cam, title, save_path):
    cols  = ["Original", "GradCAM"] + list(maps.keys())
    n     = len(cols)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    img_np = np.array(img_pil.resize((IMAGE_SIZE, IMAGE_SIZE))) / 255.0
    H, W   = img_np.shape[:2]

    for i, name in enumerate(cols):
        ax = axes[i]
        ax.imshow(img_np)
        if name != "Original":
            raw = cam if name == "GradCAM" else maps[name]
            heat = resize_map(raw, (W, H))
            ax.imshow(heat, cmap='jet', alpha=0.5)
        ax.set_title(name, fontsize=9)
        ax.axis("off")

    fig.suptitle(title, fontsize=10, y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=120)
    plt.show(fig)
    print(f"  Saved: {save_path}")


# =========================================================
# 🔥 QUANTITATIVE EXPLAINABILITY COMPARISON
#    Pearson & Spearman correlation between GradCAM and
#    each attention map (SE / WIN / AX)
# =========================================================

class ExplainabilityMetrics:
    """
    Accumulates per-sample correlation stats between GradCAM and each
    attention branch, then reports mean ± std per class.
    """
    def __init__(self):
        # {class_name: {map_name: [pearson_r, …]}}
        self.records = defaultdict(lambda: defaultdict(list))

    def update(self, cls_name, cam, maps, img_size):
        H, W = img_size
        cam_r = resize_map(cam, (W, H)).ravel()

        for mname, mdata in maps.items():
            m_r  = resize_map(mdata, (W, H)).ravel()
            pr   = pearsonr(cam_r, m_r)[0]
            sr   = spearmanr(cam_r, m_r)[0]
            self.records[cls_name][mname + "_pearson"].append(pr)
            self.records[cls_name][mname + "_spearman"].append(sr)

    def summary(self):
        rows = []
        for cls, metrics in sorted(self.records.items()):
            row = {"class": cls}
            for mname, vals in metrics.items():
                row[mname + "_mean"] = float(np.mean(vals))
                row[mname + "_std"]  = float(np.std(vals))
            rows.append(row)
        return rows

    def print_summary(self):
        print("\n" + "="*60)
        print("  QUANTITATIVE EXPLAINABILITY: GradCAM ↔ Attention Correlation")
        print("="*60)
        for row in self.summary():
            print(f"\n  Class: {row['class']}")
            for k, v in row.items():
                if k != "class":
                    print(f"    {k:35s}: {v:.4f}")

    def plot_correlation_heatmap(self, save_path):
        rows  = self.summary()
        cls_  = [r["class"] for r in rows]
        keys  = [k for k in rows[0] if k != "class" and "mean" in k]
        data  = np.array([[r[k] for k in keys] for r in rows])

        fig, ax = plt.subplots(figsize=(max(8, len(keys) * 1.5), max(4, len(cls_) * 0.6 + 1)))
        im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=-1, vmax=1)
        ax.set_xticks(range(len(keys)))
        ax.set_xticklabels([k.replace("_mean","") for k in keys], rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(len(cls_)))
        ax.set_yticklabels(cls_, fontsize=8)
        for i in range(len(cls_)):
            for j in range(len(keys)):
                ax.text(j, i, f"{data[i,j]:.2f}", ha='center', va='center', fontsize=7)
        plt.colorbar(im, ax=ax, label="Correlation (r)")
        ax.set_title("GradCAM ↔ Attention Correlation (mean per class)", fontsize=10)
        plt.tight_layout()
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        plt.show(fig)
        print(f"  Saved correlation heatmap: {save_path}")


# =========================================================
# 🔥 FLOPs + LATENCY
# =========================================================

def measure_flops_and_latency(mdl, device, img_size=224, n_runs=LATENCY_RUNS):
    dummy = torch.randn(1, 3, img_size, img_size).to(device)

    # ── FLOPs ──
    if FVCORE:
        flop_analyzer = FlopCountAnalysis(mdl, dummy)
        flop_analyzer.unsupported_ops_warnings(False)
        flop_analyzer.uncalled_modules_warnings(False)
        total_flops = flop_analyzer.total()
        param_table = parameter_count_table(mdl)
    else:
        total_flops = None
        param_table = None

    total_params = sum(p.numel() for p in mdl.parameters())
    trainable    = sum(p.numel() for p in mdl.parameters() if p.requires_grad)

    # ── Latency ──
    # warm-up
    with torch.no_grad():
        for _ in range(10):
            _ = mdl(dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            mdl(dummy)
            if device.type == "cuda":
                torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1000)   # ms

    lat_mean = float(np.mean(times))
    lat_std  = float(np.std(times))

    print("\n" + "="*60)
    print("  FLOPs & LATENCY")
    print("="*60)
    if total_flops:
        print(f"  GFLOPs        : {total_flops/1e9:.3f} G")
    print(f"  Total params  : {total_params/1e6:.3f} M")
    print(f"  Trainable     : {trainable/1e6:.3f} M")
    print(f"  Latency (bs=1): {lat_mean:.2f} ± {lat_std:.2f} ms  ({n_runs} runs)")
    if param_table:
        print("\n  Parameter breakdown:\n", param_table)

    return {
        "GFLOPs"          : round(total_flops / 1e9, 3) if total_flops else "N/A",
        "Total_params_M"  : round(total_params / 1e6, 3),
        "Trainable_M"     : round(trainable / 1e6, 3),
        "Latency_mean_ms" : round(lat_mean, 2),
        "Latency_std_ms"  : round(lat_std, 2),
    }


# =========================================================
# 🔥 ABLATION STUDY
#    Systematically ablate SE / WIN / AX by zeroing their
#    contribution and measure per-class accuracy drop.
# =========================================================

class AblationHook:
    """Zeroes out a module's output to simulate removal."""
    def __init__(self, mdl, module_name):
        self.handle = None
        mod = getattr(mdl, module_name)
        self.handle = mod.register_forward_hook(
            lambda m, i, o: torch.zeros_like(o)
        )

    def remove(self):
        if self.handle:
            self.handle.remove()

def run_ablation(mdl, loader, class_names, device):
    configs = {
        "Full model"  : [],
        "No SE"       : ["se"],
        "No WIN"      : ["win"],
        "No AX"       : ["ax"],
        "No SE+WIN"   : ["se", "win"],
        "No SE+AX"    : ["se", "ax"],
        "No WIN+AX"   : ["win", "ax"],
        "No SE+WIN+AX": ["se", "win", "ax"],
    }

    results = {}
    for cfg_name, ablate_list in configs.items():
        hooks = [AblationHook(mdl, m) for m in ablate_list]

        all_true, all_pred = [], []
        with torch.no_grad():
            for imgs, labels in loader:
                imgs   = imgs.to(device)
                out    = mdl(imgs)
                preds  = torch.argmax(out, dim=1).cpu().tolist()
                all_pred.extend(preds)
                all_true.extend(labels.tolist())

        acc = accuracy_score(all_true, all_pred) * 100
        per_class = {}
        for ci, cname in enumerate(class_names):
            idx = [i for i, t in enumerate(all_true) if t == ci]
            if idx:
                per_class[cname] = accuracy_score(
                    [all_true[i] for i in idx],
                    [all_pred[i] for i in idx]
                ) * 100
        results[cfg_name] = {"overall": acc, "per_class": per_class}

        for h in hooks:
            h.remove()

        print(f"  {cfg_name:25s}: {acc:.2f}%")

    return results

def plot_ablation(ablation_results, class_names, save_path):
    cfg_names = list(ablation_results.keys())
    overall   = [ablation_results[c]["overall"] for c in cfg_names]

    # ── bar chart – overall ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].barh(cfg_names, overall, color='steelblue')
    axes[0].axvline(overall[0], color='red', linestyle='--', label='Full model')
    axes[0].set_xlabel("Accuracy (%)")
    axes[0].set_title("Overall Accuracy per Ablation Config")
    axes[0].legend()
    axes[0].invert_yaxis()

    # ── heatmap – per-class drop ──
    base = ablation_results["Full model"]["per_class"]
    drop_matrix = []
    for cfg in cfg_names:
        row = [ablation_results[cfg]["per_class"].get(c, 0) - base.get(c, 0)
               for c in class_names]
        drop_matrix.append(row)

    dm = np.array(drop_matrix)
    im = axes[1].imshow(dm, cmap='RdYlGn', aspect='auto',
                        vmin=dm.min(), vmax=max(abs(dm.min()), abs(dm.max())))
    axes[1].set_xticks(range(len(class_names)))
    axes[1].set_xticklabels(class_names, rotation=45, ha='right', fontsize=7)
    axes[1].set_yticks(range(len(cfg_names)))
    axes[1].set_yticklabels(cfg_names, fontsize=8)
    for i in range(len(cfg_names)):
        for j in range(len(class_names)):
            axes[1].text(j, i, f"{dm[i,j]:+.1f}", ha='center', va='center', fontsize=6)
    plt.colorbar(im, ax=axes[1], label="Accuracy drop (%)")
    axes[1].set_title("Per-class Accuracy Δ vs Full Model")

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show(fig)
    print(f"  Saved ablation plot: {save_path}")


# =========================================================
# 🔥 PDF REPORT (reportlab)
# =========================================================

def build_pdf_report(save_path, flop_stats, ablation_results,
                     corr_rows, class_names, img_dir):

    doc    = SimpleDocTemplate(save_path, pagesize=A4,
                               leftMargin=2*cm, rightMargin=2*cm,
                               topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    story  = []

    # ── styles ──
    h1 = ParagraphStyle("h1", parent=styles["Heading1"], spaceAfter=6,
                         textColor=colors.HexColor("#1a237e"))
    h2 = ParagraphStyle("h2", parent=styles["Heading2"], spaceAfter=4,
                         textColor=colors.HexColor("#283593"))
    normal = styles["Normal"]
    code   = ParagraphStyle("code", parent=normal, fontName="Courier",
                             fontSize=8, leading=10,
                             backColor=colors.HexColor("#f5f5f5"))

    def hr():
        return HRFlowable(width="100%", thickness=0.5,
                          color=colors.HexColor("#9e9e9e"), spaceAfter=6)

    # ── title ──
    story.append(Paragraph("HEASNet – Advanced Interpretability Report", h1))
    story.append(hr())
    story.append(Spacer(1, 6))

    # ── 1. Model efficiency ──
    story.append(Paragraph("1. Model Efficiency (FLOPs & Latency)", h2))
    eff_data = [["Metric", "Value"]] + [
        [k.replace("_", " "), str(v)] for k, v in flop_stats.items()
    ]
    t = Table(eff_data, colWidths=[9*cm, 7*cm])
    t.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#3949ab")),
        ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
        ("FONTNAME",   (0,0), (-1,0), "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1),
         [colors.HexColor("#e8eaf6"), colors.white]),
        ("GRID", (0,0), (-1,-1), 0.3, colors.grey),
        ("FONTSIZE", (0,0), (-1,-1), 9),
        ("TOPPADDING", (0,0), (-1,-1), 4),
        ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    story.append(t)
    story.append(Spacer(1, 12))

    # ── 2. Quantitative explainability ──
    story.append(Paragraph("2. Quantitative Explainability (GradCAM ↔ Attention)", h2))
    story.append(Paragraph(
        "Pearson and Spearman correlations between the GradCAM saliency map and "
        "each attention branch (SE, WIN, AX) are computed per class. Higher values "
        "indicate that the attention mechanism focuses on the same spatial regions "
        "as gradient-based importance.", normal))
    story.append(Spacer(1, 6))

    # Table header from first row keys (minus 'class')
    if corr_rows:
        col_keys = [k for k in corr_rows[0] if k != "class"]
        hdr = ["Class"] + [k.replace("_mean","").replace("_"," ") for k in col_keys]
        corr_data = [hdr]
        for row in corr_rows:
            corr_data.append(
                [row["class"]] + [f"{row[k]:.3f}" for k in col_keys]
            )
        ncols = len(hdr)
        col_w = [16.5*cm / ncols] * ncols
        ct = Table(corr_data, colWidths=col_w)
        ct.setStyle(TableStyle([
            ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#00695c")),
            ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
            ("FONTNAME",   (0,0), (-1,0), "Helvetica-Bold"),
            ("ROWBACKGROUNDS", (0,1), (-1,-1),
             [colors.HexColor("#e0f2f1"), colors.white]),
            ("GRID", (0,0), (-1,-1), 0.3, colors.grey),
            ("FONTSIZE", (0,0), (-1,-1), 7),
            ("TOPPADDING",    (0,0), (-1,-1), 3),
            ("BOTTOMPADDING", (0,0), (-1,-1), 3),
        ]))
        story.append(ct)

    corr_img = os.path.join(img_dir, "corr_heatmap.png")
    if os.path.exists(corr_img):
        story.append(Spacer(1, 8))
        story.append(RLImage(corr_img, width=16*cm, height=6*cm))
    story.append(Spacer(1, 12))

    # ── 3. Ablation study ──
    story.append(Paragraph("3. Ablation Study", h2))
    story.append(Paragraph(
        "Each attention module (SE, WIN, AX) is individually zeroed out to measure "
        "its contribution to classification accuracy.", normal))
    story.append(Spacer(1, 6))

    abl_data = [["Configuration", "Overall Accuracy (%)"]]
    for cfg, res in ablation_results.items():
        abl_data.append([cfg, f"{res['overall']:.2f}"])
    at = Table(abl_data, colWidths=[11*cm, 5.5*cm])
    at.setStyle(TableStyle([
        ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#4527a0")),
        ("TEXTCOLOR",  (0,0), (-1,0), colors.white),
        ("FONTNAME",   (0,0), (-1,0), "Helvetica-Bold"),
        ("ROWBACKGROUNDS", (0,1), (-1,-1),
         [colors.HexColor("#ede7f6"), colors.white]),
        ("GRID", (0,0), (-1,-1), 0.3, colors.grey),
        ("FONTSIZE", (0,0), (-1,-1), 9),
        ("TOPPADDING",    (0,0), (-1,-1), 4),
        ("BOTTOMPADDING", (0,0), (-1,-1), 4),
    ]))
    story.append(at)

    abl_img = os.path.join(img_dir, "ablation.png")
    if os.path.exists(abl_img):
        story.append(Spacer(1, 8))
        story.append(RLImage(abl_img, width=16*cm, height=6*cm))
    story.append(PageBreak())

    # ── 4. Visualization gallery ──
    story.append(Paragraph("4. Grad-CAM & Attention Visualizations", h2))
    story.append(Paragraph(
        "One correct prediction and one wrong prediction per class. "
        "Overlaid: GradCAM (gradient-based) and attention maps (SE/WIN/AX).", normal))
    story.append(Spacer(1, 6))

    vis_imgs = sorted([
        f for f in os.listdir(img_dir)
        if f.endswith(".png") and f not in ("corr_heatmap.png", "ablation.png")
    ])
    for fname in vis_imgs:
        fpath = os.path.join(img_dir, fname)
        label = fname.replace("_", " ").replace(".png", "").title()
        story.append(Paragraph(label, styles["Italic"]))
        story.append(RLImage(fpath, width=16*cm, height=3.5*cm))
        story.append(Spacer(1, 6))

    doc.build(story)
    print(f"\n✅ PDF Report saved: {save_path}")


# =========================================================
# 🔥 MAIN PIPELINE
# =========================================================

# ── 1. FLOPs & Latency ──
print("\n⏱  Measuring FLOPs & Latency …")
flop_stats = measure_flops_and_latency(model, DEVICE)

# ── 2. Interpretability + Correlation ──
gradcam  = GradCAM(model)
att_cap  = AttentionCapture(model)
exp_met  = ExplainabilityMetrics()

shown_c  = {i: False for i in range(NUM_CLASSES)}
shown_w  = {i: False for i in range(NUM_CLASSES)}

print("\n🔍 Running Grad-CAM + Attention visualizations …")
for path, label in dataset.samples:

    if all(shown_c.values()) and all(shown_w.values()):
        break

    img_pil = Image.open(path).convert("RGB")
    x       = transform(img_pil).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out  = model(x)
        pred = torch.argmax(out).item()

    if pred == label and not shown_c[label]:
        tag = "correct"; shown_c[label] = True
    elif pred != label and not shown_w[label]:
        tag = "wrong";   shown_w[label] = True
    else:
        # still collect correlation data even if we don't visualise
        x_g = x.detach().requires_grad_(True)
        cam  = gradcam.generate(x_g, pred)
        maps = att_cap.get()
        exp_met.update(class_names[label], cam, maps,
                       (IMAGE_SIZE, IMAGE_SIZE))
        continue

    # needs grad for backward
    x_g  = x.detach().clone().requires_grad_(True)
    cam  = gradcam.generate(x_g, pred)
    maps = att_cap.get()

    # correlation accumulation
    exp_met.update(class_names[label], cam, maps,
                   (IMAGE_SIZE, IMAGE_SIZE))

    title = f"{tag} | GT:{class_names[label]} | Pred:{class_names[pred]}"
    save  = os.path.join(SAVE_DIR, f"{tag}_{class_names[label]}.png")
    plot_all(img_pil, maps, cam, title, save)

exp_met.print_summary()
exp_met.plot_correlation_heatmap(os.path.join(SAVE_DIR, "corr_heatmap.png"))
corr_rows = exp_met.summary()

# ── 3. Attention strength (numeric) ──
print("\n=== Attention Strength ===")
for k, v in att_cap.maps.items():
    print(f"  {k}: mean={np.mean(v):.5f}  std={np.std(v):.5f}")

# ── 4. Ablation study ──
print("\n🧪 Running Ablation Study …")
loader = torch.utils.data.DataLoader(dataset, batch_size=32,
                                     shuffle=False, num_workers=2)
ablation_results = run_ablation(model, loader, class_names, DEVICE)
plot_ablation(ablation_results, class_names,
              os.path.join(SAVE_DIR, "ablation.png"))

# ── 5. PDF report ──
print("\n📄 Building PDF report …")
pdf_path = os.path.join(SAVE_DIR, "HEASNet_Interpretability_Report.pdf")
build_pdf_report(
    save_path       = pdf_path,
    flop_stats      = flop_stats,
    ablation_results= ablation_results,
    corr_rows       = corr_rows,
    class_names     = class_names,
    img_dir         = SAVE_DIR,
)

print("\n🎉 ALL DONE!")
print(f"   Visuals + Report → {SAVE_DIR}/")

In [ ]:
import shutil

shutil.make_archive('/kaggle/working/interpretability', 'zip', '/kaggle/working/interpretability')